[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S14_matplotlib_anatomia.ipynb)

# Sesión 14 · Matplotlib: anatomía de un gráfico

**Módulo 4: Matplotlib** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Crear un gráfico con `fig, ax = plt.subplots()` y explicar qué es cada objeto.
2. Elegir entre `plot`, `bar`, `barh`, `hist`, `scatter` y `boxplot` según la pregunta.
3. Poner título y etiquetas a los ejes.
4. Formatear los ejes con montos en soles y porcentajes.

## 📋 Qué debes saber antes
Módulos 2 y 3: arrays de NumPy, Series y DataFrames de pandas.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`. Usa los nombres de variables que se piden (`fig1`, `ax1`...): el verificador revisa esos objetos.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Importa Matplotlib como `plt`, genera los datos, aplica un estilo sobrio a todos los gráficos y carga las funciones que revisan tus respuestas.

El estilo deja un fondo claro, quita los bordes de arriba y de la derecha, usa una grilla tenue y asigna los colores en un orden fijo pensado para que se distingan también con daltonismo. Los colores quedan disponibles por nombre: `AZUL`, `NARANJA`, `AQUA`... y `GRIS` para lo secundario.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, aplica un estilo sobrio a los gráficos y carga los verificadores.
import copy
import hashlib
import math
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, MultipleLocator, PercentFormatter, StrMethodFormatter

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})

# ---------- Datos de práctica: ventas de tiendas ----------
meses = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "set", "oct", "nov", "dic"]
_estacion = np.array([0.9, 0.85, 0.95, 1.0, 1.0, 1.05, 1.1, 1.0, 0.95, 1.0, 1.1, 1.35])
ventas_mes = pd.Series(np.round((180_000 + np.arange(12) * 3000) * _estacion * rng.uniform(0.95, 1.05, 12), -2), index=meses)
ventas_tienda = pd.Series(np.round(rng.uniform(90_000, 240_000, 5), -3),
                          index=["Barranco", "Lince", "Miraflores", "San Isidro", "Surco"])
tickets = np.round(rng.lognormal(4.2, 0.5, 500), 2)
tickets_app = np.round(rng.lognormal(4.0, 0.45, 300), 2)
tickets_tienda = np.round(rng.lognormal(4.4, 0.5, 300), 2)

# ---------- Datos de práctica: clientes y operaciones de un banco ----------
_ingreso = np.round(np.clip(rng.normal(4500, 1500, 200), 1200, None), 2)
clientes = pd.DataFrame({"ingreso": _ingreso, "gasto": np.round(np.clip(0.55 * _ingreso + rng.normal(0, 500, 200), 300, None), 2)})
canales = ["agencia", "app", "teléfono"]
tiempos = {
    "agencia": np.round(np.concatenate([rng.gamma(4, 5, 78), [65.0, 72.0]]), 1),
    "app": np.round(rng.gamma(2, 1.5, 120), 1),
    "teléfono": np.round(rng.gamma(3, 3, 60), 1),
}
morosidad = pd.Series(np.round(rng.uniform(0.02, 0.06, 12), 4), index=meses)
saldo_diario = pd.Series(np.round(1500 + np.cumsum(rng.normal(-25, 260, 62)), 2),
                         index=pd.date_range("2026-07-01", periods=62, freq="D"))

_D = copy.deepcopy({k: globals()[k] for k in ["ventas_mes", "ventas_tienda", "tickets", "tickets_app", "tickets_tienda",
                                               "clientes", "tiempos", "morosidad", "saldo_diario"]})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")


def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    f = r.var("fig1")
    if f is not _FALTA and not isinstance(f, mpl.figure.Figure):
        r.mal("`fig1` debería ser la figura que devuelve `plt.subplots()`.")
    ax = _grafico(r, "ax1")
    if ax is not None:
        lineas = ax.get_lines()
        v = _D["ventas_mes"]
        if len(lineas) != 1:
            r.mal(f"`ax1` tiene {len(lineas)} líneas y se esperaba 1.")
        elif not _cerca_lista(lineas[0].get_ydata(), v.tolist()):
            r.mal("La línea de `ax1` no tiene las ventas mensuales en el eje y.")
        elif [str(x) for x in lineas[0].get_xdata()] != list(v.index):
            r.mal("La línea de `ax1` debería usar los nombres de los meses en el eje x.")
        else:
            r.ok("La línea de `ax1` tiene los datos correctos.")
            if lineas[0].get_marker() in (None, "None", "", " "):
                r.mal("Agrega un marcador a la línea (por ejemplo, `marker=\"o\"`) para que se vea cada mes.")
        _rotulos(r, "ax1", ax, "Ventas mensuales 2025", "Mes", "Ventas (S/)")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_tipo_subplots": "6b88ca3380698534b1380b457e554090cca21744b910883712503cb72f5e473a",
        "pred_n_lineas": "b4d4f68c2268549cada66d24ae3a1902d4152a98ad614bbf891edf235ef99218",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    v = _D["ventas_tienda"]
    desc = sorted(v.items(), key=lambda kv: -kv[1])
    for nombre, orden, medida in (("ax2", desc, "get_height"), ("ax3", desc[::-1], "get_width")):
        ax = _grafico(r, nombre)
        if ax is None:
            continue
        barras = _barras(ax)
        if len(barras) != len(v):
            r.mal(f"`{nombre}` tiene {len(barras)} barras y se esperaban {len(v)}.")
            continue
        tipo = "verticales" if medida == "get_height" else "horizontales"
        largos = [getattr(b, medida)() for b in barras]
        if not _cerca_lista(largos, [x for _, x in orden]):
            r.mal(f"Las barras de `{nombre}` no tienen los valores en el orden pedido; revisa que sean barras {tipo} y el orden.")
        elif len({_hex(b.get_facecolor()) for b in barras}) != 1:
            r.mal(f"Las barras de `{nombre}` tienen colores distintos: una sola serie va en un solo color.")
        else:
            r.ok(f"Las barras de `{nombre}` son correctas.")
    ax2, ax3 = globals().get("ax2"), globals().get("ax3")
    if isinstance(ax2, mpl.axes.Axes):
        _rotulos(r, "ax2", ax2, "Ventas por tienda, setiembre", None, "Ventas (S/)")
    if isinstance(ax3, mpl.axes.Axes):
        _rotulos(r, "ax3", ax3, "Ventas por tienda, setiembre", "Ventas (S/)", None)
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_barh_arriba": "f5223646e72facdaf7d47e9f8ea168e490e4261c641e2788f1672df9f9574d86",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3")
    ax = _grafico(r, "ax4")
    if ax is not None:
        barras = _barras(ax)
        t = _D["tickets"].tolist()
        if len(barras) != 20:
            r.mal(f"El histograma de `ax4` tiene {len(barras)} barras y se pidieron 20 intervalos.")
        elif round(sum(b.get_height() for b in barras)) != len(t):
            r.mal("Las barras del histograma no suman la cantidad de tickets: ¿usaste `tickets`?")
        else:
            r.ok("El histograma de `ax4` es correcto.")
        media = statistics.fmean(t)
        verticales = [l for l in ax.get_lines() if len(set(map(float, l.get_xdata()))) == 1]
        if any(abs(float(l.get_xdata()[0]) - media) < 1e-6 for l in verticales):
            r.ok("La línea vertical marca el ticket promedio.")
        else:
            r.mal("Falta una línea vertical en el ticket promedio (investiga `ax.axvline`).")
        _rotulos(r, "ax4", ax, "Distribución del ticket", "Ticket (S/)", "Cantidad de ventas")
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4")
    ax = _grafico(r, "ax5")
    c = _D["clientes"]
    if ax is not None:
        puntos = [col for col in ax.collections if isinstance(col, mpl.collections.PathCollection)]
        if len(puntos) != 1:
            r.mal("`ax5` debería tener un solo gráfico de dispersión (`ax.scatter`).")
        else:
            xy = puntos[0].get_offsets()
            if len(xy) == len(c) and _cerca_lista(xy[:, 0], c["ingreso"].tolist()) and _cerca_lista(xy[:, 1], c["gasto"].tolist()):
                r.ok("Los puntos de `ax5` son correctos.")
            else:
                r.mal("Los puntos de `ax5` deberían tener el ingreso en el eje x y el gasto en el eje y.")
        _rotulos(r, "ax5", ax, "Ingreso y gasto de los clientes", "Ingreso mensual (S/)", "Gasto mensual (S/)")
    ax = _grafico(r, "ax6")
    if ax is not None:
        etiquetas = [t.get_text() for t in ax.get_xticklabels()]
        horizontales = [float(l.get_ydata()[0]) for l in ax.get_lines() if len(l.get_ydata()) == 2 and l.get_ydata()[0] == l.get_ydata()[1]]
        medianas = [statistics.median(_D["tiempos"][k].tolist()) for k in ("agencia", "app", "teléfono")]
        if etiquetas != ["agencia", "app", "teléfono"]:
            r.mal("Las etiquetas del eje x de `ax6` deberían ser los tres canales, en el orden de `canales`.")
        elif not all(any(abs(m - h) < 1e-6 for h in horizontales) for m in medianas):
            r.mal("Las cajas de `ax6` no corresponden a los tiempos de cada canal, en el orden de `canales`.")
        else:
            r.ok("El diagrama de caja de `ax6` es correcto.")
        _rotulos(r, "ax6", ax, "Tiempo de atención por canal", None, "Minutos de atención")
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5")
    ax = _grafico(r, "ax7")
    if ax is not None:
        linea = ax.get_lines()
        if len(linea) != 1 or not _cerca_lista(linea[0].get_ydata(), _D["morosidad"].tolist()):
            r.mal("`ax7` debería tener una sola línea con la tasa de morosidad.")
        else:
            r.ok("La línea de `ax7` es correcta.")
        etiqueta = _formato(ax, "y", 0.05)
        r.ok("El eje y de `ax7` muestra porcentajes con 1 decimal.") if etiqueta == "5.0%" else \
            r.mal(f"En `ax7`, el valor 0.05 se muestra como {etiqueta!r} y debería verse como porcentaje con 1 decimal.")
        abajo, arriba = ax.get_ylim()
        r.ok("El eje y de `ax7` va de 0 a 8 %.") if abs(abajo) < 1e-9 and abs(arriba - 0.08) < 1e-9 else \
            r.mal(f"El eje y de `ax7` va de {abajo:.3f} a {arriba:.3f}; debería ir de 0 a 0.08.")
        _rotulos(r, "ax7", ax, "Tasa de morosidad mensual", None, None)
    ax = _grafico(r, "ax8")
    if ax is not None:
        barras = _barras(ax)
        if len(barras) != 5:
            r.mal("`ax8` debería tener 5 barras horizontales, una por tienda.")
        etiqueta = _formato(ax, "x", 150_000)
        r.ok("El eje x de `ax8` muestra los montos en miles.") if etiqueta == "S/ 150 mil" else \
            r.mal(f"En `ax8`, el valor 150000 se muestra como {etiqueta!r} y debería verse como \"S/ 150 mil\".")
        marcas = [float(x) for x in ax.get_xticks()]
        pasos = {round(b - a) for a, b in zip(marcas, marcas[1:])}
        r.ok("Las marcas del eje x de `ax8` van cada 50 mil.") if pasos == {50_000} else \
            r.mal("Las marcas del eje x de `ax8` deberían ir cada 50 000 (investiga `MultipleLocator`).")
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    ax = _grafico(r, "ax_saldo")
    s = _D["saldo_diario"]
    if ax is not None:
        largas = [l for l in ax.get_lines() if len(l.get_ydata()) == len(s)]
        if not largas or not _cerca_lista(largas[0].get_ydata(), s.tolist()):
            r.mal("`ax_saldo` debería tener una línea con el saldo de cada día.")
        else:
            r.ok("La línea del saldo es correcta.")
        horiz = [l for l in ax.get_lines() if len(set(map(float, l.get_ydata()))) == 1 and float(l.get_ydata()[0]) == 0.0]
        r.ok("Hay una línea de referencia en 0.") if horiz else r.mal("Falta una línea horizontal en 0 (investiga `ax.axhline`).")
        puntos = [c for c in ax.collections if isinstance(c, mpl.collections.PathCollection)]
        x_min, y_min = mdates.date2num(s.idxmin()), float(s.min())
        if any(len(p.get_offsets()) == 1 and abs(float(p.get_offsets()[0][0]) - x_min) < 1e-6
               and abs(float(p.get_offsets()[0][1]) - y_min) < 1e-6 for p in puntos):
            r.ok("El punto marca el día de saldo mínimo.")
        else:
            r.mal("Falta un punto (`ax.scatter`) en el día y el valor del saldo mínimo.")
        etiqueta = _formato(ax, "y", 2500)
        r.ok("El eje y muestra montos en soles.") if etiqueta == "S/ 2,500" else \
            r.mal(f"En el eje y, 2500 se muestra como {etiqueta!r} y debería verse como \"S/ 2,500\".")
        _rotulos(r, "ax_saldo", ax, "Saldo diario de la cuenta", None, "Saldo (S/)")
    _esc(r, "n_dias_negativos", sum(1 for x in s.tolist() if x < 0), "cuenta los días con saldo menor que 0")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    ax = _grafico(r, "ax_pro")
    if ax is not None:
        barras = _barras(ax)
        leyenda = ax.get_legend()
        etiquetas = [t.get_text() for t in leyenda.get_texts()] if leyenda else []
        if len(barras) != 60:
            r.mal(f"`ax_pro` tiene {len(barras)} barras; con dos histogramas de 30 intervalos deberían ser 60.")
        elif round(sum(b.get_height() for b in barras)) != sum(1 for x in _D["tickets_app"].tolist() + _D["tickets_tienda"].tolist() if 0 <= x <= 450):
            r.mal("Los dos histogramas deberían sumar todos los tickets de app y de tienda.")
        elif sorted(etiquetas) != ["app", "tienda"]:
            r.mal("Falta una leyenda que diga \"app\" y \"tienda\".")
        else:
            r.ok("Los dos histogramas y la leyenda son correctos.")
    r.fin()


print("✅ Setup listo. Datos generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
- `ventas_mes`: ventas de cada mes de 2025 (Series con los meses como índice).
- `ventas_tienda`: ventas de setiembre por tienda.
- `tickets`, `tickets_app`, `tickets_tienda`: importes de ventas individuales.
- `clientes`: ingreso y gasto mensual de 200 clientes de un banco.
- `tiempos`: minutos de atención por canal (un array por canal) y `canales`, la lista de canales.
- `morosidad`: tasa de morosidad mensual (0.05 es 5 %).
- `saldo_diario`: el saldo de una cuenta cada día de julio y agosto de 2026.

In [ ]:
print(ventas_mes.head(3), "\n")
print(ventas_tienda, "\n")
print(clientes.head(3), "\n")
print({k: v[:5] for k, v in tiempos.items()}, "\n")
print(saldo_diario.head(3))

---
## 1. `fig` y `ax`: el lienzo y el gráfico

### 📘 Concepto
En Matplotlib hay dos objetos principales:
- La **figura** (`fig`): el lienzo completo, la imagen que se muestra o se guarda.
- El **eje** (`ax`, de *Axes*): un gráfico dentro de la figura, con su área de dibujo, sus ejes x e y, su título y sus etiquetas.

`plt.subplots()` crea los dos a la vez y los devuelve en una tupla. Después se dibuja **sobre el eje**:

```python
fig, ax = plt.subplots(figsize=(8, 4))   # ancho y alto en pulgadas
ax.plot(x, y)                            # una línea
ax.set_title("...")                      # título
ax.set_xlabel("...")                     # etiqueta del eje x
ax.set_ylabel("...")                     # etiqueta del eje y
```

`ax.plot` dibuja una línea: sirve para mostrar cómo cambia algo en el tiempo. Cada llamada a `ax.plot` agrega una línea más al mismo eje. Con `marker="o"` se marca cada punto. Colab muestra el gráfico al terminar la celda.

In [ ]:
dias_ej = ["lun", "mar", "mié", "jue", "vie"]
visitas_ej = [120, 340, 90, 410, 205]

fig_ej, ax_ej = plt.subplots(figsize=(6, 3))
ax_ej.plot(dias_ej, visitas_ej, marker="o")
ax_ej.set_title("Visitas a la tienda por día")
ax_ej.set_xlabel("Día")
ax_ej.set_ylabel("Visitas")
print(type(fig_ej).__name__, type(ax_ej).__name__, len(ax_ej.get_lines()))

### ✍️ Tu turno · Ejercicio 1: la línea de ventas del año
**Parte A.** Crea `fig1, ax1` con un tamaño de 8 × 4 y dibuja las ventas mensuales de `ventas_mes` (meses en el eje x) con un marcador en cada punto. Ponle:
- título: `Ventas mensuales 2025`
- eje x: `Mes`
- eje y: `Ventas (S/)`

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_tipo_subplots` | ¿de qué tipo es lo que devuelve `plt.subplots()`? | el nombre del tipo, como texto |
| `pred_n_lineas` | si llamas dos veces a `ax.plot(...)` sobre el mismo eje, ¿cuánto da `len(ax.get_lines())`? | número |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

`ventas_mes.index` son los meses y `ventas_mes.values`, las ventas.
</details>

<details><summary>💡 Pista 2</summary>

`ax1.plot(ventas_mes.index, ventas_mes.values, marker="o")`, y después las tres llamadas `set_...`.
</details>

---
## 2. Barras: `bar` y `barh`

### 📘 Concepto
Las barras comparan **cantidades entre categorías**. Buenas prácticas:
- **Ordena** las barras por valor: el orden hace el trabajo de comparar por el lector.
- **Una serie, un color**: pintar cada barra de un color distinto no agrega información y distrae.
- Las barras **siempre empiezan en 0**: si el eje empieza en otro valor, las diferencias se ven exageradas.
- Con nombres largos o muchas categorías, usa **barras horizontales** (`barh`): los nombres se leen sin girar la cabeza. En `barh`, la primera categoría queda **abajo**; para tener la mayor arriba, ordena de menor a mayor.

In [ ]:
productos_ej = pd.Series({"polo": 320, "jean": 150, "gorra": 90, "casaca": 210})
orden_ej = productos_ej.sort_values(ascending=False)

fig_ej, ax_ej = plt.subplots(figsize=(6, 3))
ax_ej.bar(orden_ej.index, orden_ej.values)
ax_ej.set_title("Unidades vendidas por producto")

fig_ej2, ax_ej2 = plt.subplots(figsize=(6, 3))
ax_ej2.barh(productos_ej.sort_values().index, productos_ej.sort_values().values)
ax_ej2.set_title("Lo mismo, en horizontal (la mayor arriba)")

### ✍️ Tu turno · Ejercicio 2: ventas por tienda
**Parte A.**
1. `fig2, ax2`: barras **verticales** con las ventas de `ventas_tienda`, de la mayor a la menor. Título `Ventas por tienda, setiembre` y eje y `Ventas (S/)`.
2. `fig3, ax3`: las mismas ventas en barras **horizontales**, con la mayor **arriba**. El mismo título y eje x `Ventas (S/)`.

**Parte B.** Predice **sin ejecutar**: en `ax.barh(["a", "b", "c"], [1, 2, 3])`, ¿qué categoría queda arriba? Responde en `pred_barh_arriba` con la letra.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Ordena primero con `sort_values`. Para las verticales, de mayor a menor (`ascending=False`); para las horizontales, de menor a mayor.
</details>

<details><summary>💡 Pista 2</summary>

`orden = ventas_tienda.sort_values(ascending=False)` y `ax2.bar(orden.index, orden.values)`. En `barh`, el primer valor se dibuja abajo.
</details>

---
## 3. Histograma: cómo se reparten los valores

### 📘 Concepto
Un **histograma** (`ax.hist(datos, bins=n)`) divide el rango de los datos en `n` intervalos y dibuja cuántos valores caen en cada uno. Responde "¿cómo se distribuyen?": dónde se concentran, si hay valores extremos, si la distribución es simétrica.

Para marcar una referencia, como el promedio, se usa una línea vertical: `ax.axvline(valor, color=..., linewidth=...)`. Una línea fina y oscura (`TINTA_2`) marca la referencia sin competir con los datos.

In [ ]:
gen_ej = np.random.default_rng(1)
esperas_ej = gen_ej.exponential(5, 300)

fig_ej, ax_ej = plt.subplots(figsize=(6, 3))
ax_ej.hist(esperas_ej, bins=15)
ax_ej.axvline(esperas_ej.mean(), color=TINTA_2, linewidth=1.5)
ax_ej.set_title("Minutos de espera en caja")

### ✍️ Tu turno · Ejercicio 3: el ticket de venta
Crea `fig4, ax4` con el histograma de `tickets` en **20** intervalos y una línea vertical en el **ticket promedio** (color `TINTA_2`, grosor 1.5). Ponle:
- título: `Distribución del ticket`
- eje x: `Ticket (S/)`
- eje y: `Cantidad de ventas`

Mira el gráfico: ¿el promedio está en el centro de la distribución o se corre hacia algún lado? ¿Por qué?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

`ax4.hist(tickets, bins=20)` y `ax4.axvline(...)`.
</details>

<details><summary>💡 Pista 2</summary>

El promedio es `tickets.mean()`. Si la distribución tiene una cola larga a la derecha, el promedio queda a la derecha del pico.
</details>

---
## 4. Relación entre dos variables y comparación de distribuciones

### 📘 Concepto
- **`ax.scatter(x, y)`**, el gráfico de dispersión: un punto por observación. Muestra si dos variables se mueven juntas. Con muchos puntos, usa puntos pequeños (`s=20`) y algo de transparencia (`alpha=0.6`) para ver dónde se amontonan.
- **`ax.boxplot(lista_de_arrays)`**, el diagrama de caja: resume cada grupo con su mediana (la línea de la caja), el 50 % central (la caja), el rango habitual (los bigotes) y los valores atípicos (los puntos sueltos). Sirve para comparar distribuciones de varios grupos lado a lado. Los grupos quedan en las posiciones 1, 2, 3...; sus nombres se ponen con `ax.set_xticks([1, 2, 3], nombres)`.

In [ ]:
gen_ej = np.random.default_rng(2)
horas_ej = gen_ej.uniform(1, 10, 50)
nota_ej = 10 + 1.2 * horas_ej + gen_ej.normal(0, 1.5, 50)

fig_ej, ax_ej = plt.subplots(figsize=(5, 3))
ax_ej.scatter(horas_ej, nota_ej, s=20, alpha=0.6)
ax_ej.set_title("Horas de estudio y nota")

fig_ej2, ax_ej2 = plt.subplots(figsize=(5, 3))
ax_ej2.boxplot([gen_ej.normal(20, 3, 40), gen_ej.normal(25, 6, 40)])
ax_ej2.set_xticks([1, 2], ["turno mañana", "turno tarde"])

### ✍️ Tu turno · Ejercicio 4: clientes y tiempos de atención
1. `fig5, ax5`: gráfico de dispersión con el ingreso de `clientes` en el eje x y el gasto en el eje y (`s=20`, `alpha=0.6`). Título `Ingreso y gasto de los clientes`, eje x `Ingreso mensual (S/)` y eje y `Gasto mensual (S/)`.
2. `fig6, ax6`: diagrama de caja con los tiempos de cada canal, en el orden de `canales`, con los nombres de los canales en el eje x. Título `Tiempo de atención por canal` y eje y `Minutos de atención`.

¿Qué canal atiende más rápido? ¿Cuál tiene más variación y más casos extremos?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Las columnas de `clientes` se pasan directo a `scatter`. Para el diagrama de caja, arma la lista de arrays recorriendo `canales`.
</details>

<details><summary>💡 Pista 2</summary>

`ax6.boxplot([tiempos[c] for c in canales])` y luego `ax6.set_xticks([1, 2, 3], canales)`.
</details>

---
## 5. Formato de números y porcentajes en los ejes

### 📘 Concepto
Un eje que muestra `0.05` cuando quiere decir "5 %", o `150000` cuando quiere decir "S/ 150 mil", obliga al lector a traducir. El **formateador** de un eje decide cómo se escribe cada número:

| Formateador | Ejemplo | Resultado |
|---|---|---|
| `PercentFormatter(xmax=1, decimals=1)` | 0.05 | `5.0%` |
| `StrMethodFormatter("S/ {x:,.0f}")` | 2500 | `S/ 2,500` |
| `FuncFormatter(lambda x, pos: f"S/ {x / 1000:.0f} mil")` | 150000 | `S/ 150 mil` |

Se aplican con `ax.yaxis.set_major_formatter(...)` o `ax.xaxis.set_major_formatter(...)`. En `FuncFormatter`, la función recibe el valor y su posición, y devuelve el texto.

Para fijar el rango de un eje: `ax.set_ylim(abajo, arriba)`. Una tasa se lee mejor con el eje desde 0.

Si las etiquetas se enciman (mira el eje x de `ax3` en el ejercicio 2), pon menos marcas: `ax.xaxis.set_major_locator(MultipleLocator(paso))` deja una marca cada `paso` unidades.

In [ ]:
cumplimiento_ej = pd.Series([0.82, 0.91, 0.87, 0.95], index=["T1", "T2", "T3", "T4"])
fig_ej, ax_ej = plt.subplots(figsize=(5, 3))
ax_ej.plot(cumplimiento_ej.index, cumplimiento_ej.values, marker="o")
ax_ej.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
ax_ej.set_ylim(0, 1)

fig_ej2, ax_ej2 = plt.subplots(figsize=(5, 3))
ax_ej2.bar(["A", "B"], [1_250_000, 2_300_000])
ax_ej2.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"S/ {x / 1_000_000:.1f} M"))

### ✍️ Tu turno · Ejercicio 5: ejes que se leen solos
1. `fig7, ax7`: una línea con la tasa de `morosidad` de cada mes, con el eje y en porcentaje con 1 decimal y de 0 a 0.08. Título `Tasa de morosidad mensual`.
2. `fig8, ax8`: barras horizontales con `ventas_tienda` (la mayor arriba), con el eje x escrito en miles, como `S/ 150 mil`, y una marca cada 50 000.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Copia los formateadores de la tabla. El de miles es un `FuncFormatter` con una `lambda`.
</details>

<details><summary>💡 Pista 2</summary>

`ax7.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=1))` y `ax7.set_ylim(0, 0.08)`. En `ax8`, el formateador y el `MultipleLocator(50_000)` van en `ax8.xaxis`.
</details>

---
## 🏋️ Reto final: el saldo de una cuenta
Con `saldo_diario`, crea `fig_saldo, ax_saldo` (tamaño 9 × 4) con:
1. Una línea con el saldo de cada día (las fechas en el eje x).
2. Una línea horizontal de referencia en 0, fina y gris (`GRIS`).
3. Un punto (`ax.scatter`) en el día de **saldo mínimo**, de color `NARANJA`, dibujado por encima de la línea (`zorder=3`).
4. El eje y escrito en soles, como `S/ 2,500`.
5. Título `Saldo diario de la cuenta` y eje y `Saldo (S/)`. Para que las fechas no se encimen, usa `fig_saldo.autofmt_xdate()`.

Después, `n_dias_negativos`: cuántos días cerró la cuenta con saldo negativo.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

`saldo_diario.idxmin()` da la fecha del mínimo y `saldo_diario.min()`, el valor.
</details>

<details><summary>💡 Pista 2</summary>

`ax_saldo.axhline(0, color=GRIS, linewidth=1)` para la referencia; `ax_saldo.scatter([fecha], [valor], color=NARANJA, zorder=3)` para el punto. El formato es `StrMethodFormatter("S/ {x:,.0f}")`.
</details>

---
## 🚀 Nivel pro (opcional): dos distribuciones en un mismo gráfico
Crea `fig_pro, ax_pro` con dos histogramas superpuestos: `tickets_app` y `tickets_tienda`, con los **mismos** 30 intervalos para los dos (`bins=np.linspace(0, 450, 31)`), transparencia `alpha=0.6` y una leyenda que diga `app` y `tienda` (usa `label=` en cada `hist` y luego `ax_pro.legend()`). ¿Por qué es importante que los dos usen los mismos intervalos?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar la diferencia entre la figura y el eje.
- [ ] Crear un gráfico con `plt.subplots()` y ponerle título y etiquetas.
- [ ] Elegir el tipo de gráfico según la pregunta: evolución (línea), comparación (barras), distribución (histograma o caja), relación (dispersión).
- [ ] Ordenar barras y explicar por qué van en un solo color y desde 0.
- [ ] Explicar qué muestra cada parte de un diagrama de caja.
- [ ] Marcar una referencia con `axvline` o `axhline`.
- [ ] Escribir los ejes en soles, miles o porcentajes con un formateador.

**Próxima sesión (S15):** gráficos que comunican: varios gráficos en una figura, pandas, anotaciones, color con intención y guardar en buena resolución.